# broadcasting-rules — ex9: silent-broadcast trap — catch it with a value check

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcasting-rules`. Running the final beacon cell reports progress against the `Numpy: Vectorization and broadcasting` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Vectorization and broadcasting` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcasting-rules`** (exercise 9). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcasting-rules"
DD_SUBTOPIC = "Numpy: Vectorization and broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Broadcasting — quick refresher

**The rule** (NumPy & PyTorch agree):
1. Right-align both shapes; left-pad the shorter with 1s.
2. For each pair of aligned axes: equal → keep; one is 1 → use the other; otherwise → incompatible.

**The dangerous case.** When a shape *almost* matches you can get an unintended broadcast that runs silently and produces wrong values. Always shape-check (`print(x.shape, y.shape, (x*y).shape)`) when wiring up a new pipeline.

### Exercise 9 — silent-broadcast trap — catch it with a value check

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Evaluate
> LO: Recognize when broadcasting succeeds shape-wise but applies the wrong values, and fix it with an axis insertion.
> Keywords: silent-bug, shape-trace, unsqueeze, value-spot-check, NCHW
> ```

**KCs targeted:** `detect-silent-broadcast`, `axis-insertion-via-unsqueeze`, `value-vs-shape-debugging`

**The trap.** A buggy normalizer is shipping in production. The author wrote it for NCHW images (`(B, C, H, W)`) but treated the per-channel mean `mu` of shape `(C,)` like in the NHWC case — `X - mu`. The shapes 'work' (no exception) because `H == C` in their tests by accident. On real data with `H != C` it explodes; on `H == C` data it silently subtracts the wrong values.

Implement `ex9_normalize_nchw(X, mu)` to subtract `mu` (shape `(C,)`) from each channel of `X` (shape `(B, C, H, W)`) **correctly**, regardless of whether `H == C` or not. The fix is one `unsqueeze` away — but you have to know which axes need the size-1 padding.

The test cell constructs a deliberately-trapped case where `B = 1, C = 4, H = 4, W = 6`. The buggy expression `X - mu` would not raise (because the last axis is `W = 6`, broadcasting `mu = (4,)` would fail with `W=6 ≠ C=4`, but a sloppy 'fix' like `X - mu[None, :, None, None]` might be miswritten as `X - mu[None, None, None, :]` and pass the shape check while computing nonsense). Your job: pick the right axis insertion and verify with a value check.

In [ ]:
def ex9_normalize_nchw(X: Tensor, mu: Tensor) -> Tensor:
    """Subtract per-channel mean `mu` (C,) from NCHW image batch `X` (B, C, H, W).

    The fix is one axis insertion. Find it.
    """
    raise NotImplementedError()


def _test_ex9():
    # Case 1 — H != C, the broken version would raise, so any shape-OK answer is also value-OK
    B, C, H, W = 2, 3, 5, 7
    X = t.randn(B, C, H, W)
    mu = t.tensor([10.0, 100.0, 1000.0])
    Y = ex9_normalize_nchw(X, mu)
    assert Y.shape == (B, C, H, W), f'shape mismatch: {tuple(Y.shape)}'

    # Per-channel shift must be exactly -mu[c]
    for c in range(C):
        diff = Y[:, c] - X[:, c]
        assert t.allclose(diff, t.full_like(diff, -mu[c]), atol=1e-5), (
            f'channel {c}: expected shift {-mu[c]} but range was '
            f'[{diff.min().item():.4f}, {diff.max().item():.4f}]'
        )

    # Case 2 — the trap: H == C == 4. The wrong axis insertion would pass the shape check
    # (because broadcasting (1,1,1,4) over (B,4,4,W) succeeds when W == 4) but ALSO pass
    # a naive per-image total check. We catch it by checking per-channel slices.
    B, C, H, W = 1, 4, 4, 4
    X = t.zeros(B, C, H, W)
    mu = t.tensor([1.0, 2.0, 3.0, 4.0])
    Y = ex9_normalize_nchw(X, mu)
    assert Y.shape == (B, C, H, W)
    # Every spatial location of channel c must equal -mu[c]
    for c in range(C):
        sl = Y[0, c]
        assert (sl == -mu[c]).all(), (
            f'TRAP: channel {c} got values {sl.unique().tolist()} but should be all {-mu[c]}.\n'
            f'You probably inserted axes on the wrong side — mu[None, None, None, :] '
            f'broadcasts mu over the *width* axis, not the *channel* axis.'
        )

    # Case 3 — value spot-check on a structured input so any wrong-axis bug is visible
    B, C, H, W = 1, 2, 3, 5
    X = t.arange(B * C * H * W, dtype=t.float32).reshape(B, C, H, W)
    mu = t.tensor([0.0, 100.0])
    Y = ex9_normalize_nchw(X, mu)
    # Channel 0 unchanged, channel 1 shifted by -100
    assert t.equal(Y[0, 0], X[0, 0]), 'channel 0 must be unchanged (mu=0)'
    assert t.equal(Y[0, 1], X[0, 1] - 100), 'channel 1 must be shifted by -100'
    _dd_passed.add('ex9')
    print("ex9 ✓")

_test_ex9()

<details><summary>Solution</summary>

```python
def ex9_normalize_nchw(X: Tensor, mu: Tensor) -> Tensor:
    # X is (B, C, H, W). mu is (C,). We need mu to broadcast over the C-axis only,
    # which means inserting size-1 axes for B (leading), H, and W (trailing).
    # mu[None, :, None, None] gives shape (1, C, 1, 1) → right-aligns with (B, C, H, W).
    return X - mu[None, :, None, None]
```

**The lesson.** When the broadcast result has the *right shape* but *wrong values*, only a value spot-check catches it. The two value checks the test runs:
1. Per-channel uniformity (Case 1, Case 2) — every spatial cell of channel `c` should have shifted by exactly `-mu[c]`. A wrong axis insertion produces a striped pattern instead.
2. Structured input (Case 3) — when `X = arange(...)`, *any* wrong-axis broadcast leaves a visible non-monotonic artifact in the output.

**Rule of thumb.** When you `unsqueeze` to fix a broadcast, count axes from the *target tensor*, not from the vector. Here the channel axis of `X` is at index 1, so `mu` needs size-1s in positions 0, 2, 3 — giving `mu[None, :, None, None]`. Alternative spellings: `mu.view(1, -1, 1, 1)`, `mu.reshape(1, C, 1, 1)`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex9',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()